<a href="https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii"
REPO_DIR = "flyrank-ml-internship-hadii"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

Loaded: (30000, 44)


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Five key fields, looked at before touching any correlation: `impressions_90d`,
`engagement_rate`, `scroll_rate`, `word_count`, `days_since_last_update`. All the
traffic-like ones are heavy-tailed — a few giant pages, a long tail of small ones —
so mean and median disagree, and any correlation I run later needs `log1p()` or
rank-based comparison, not raw Pearson.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for col in ["impressions_90d", "engagement_rate", "scroll_rate", "word_count", "days_since_last_update"]:
    s = df[col].dropna()
    print(f"{col}: n={len(s)}, mean={s.mean():.2f}, median={s.median():.2f}, "
          f"p95={s.quantile(0.95):.2f}, max={s.max():.2f}")

print()
print("impressions_90d, mean vs median after log1p (should move much closer together):")
print("  raw     mean={:.0f}  median={:.0f}".format(df["impressions_90d"].mean(), df["impressions_90d"].median()))
print("  log1p   mean={:.2f}  median={:.2f}".format(
    np.log1p(df["impressions_90d"]).mean(), np.log1p(df["impressions_90d"]).median()))

impressions_90d: n=30000, mean=5200.37, median=731.00, p95=22996.50, max=517715.00
engagement_rate: n=30000, mean=2.53, median=0.00, p95=12.50, max=100.00
scroll_rate: n=29875, mean=18.21, median=5.00, p95=100.00, max=300.00
word_count: n=22301, mean=3107.76, median=2877.00, p95=6173.00, max=9546.00
days_since_last_update: n=30000, mean=46.10, median=20.00, p95=104.00, max=373.00

impressions_90d, mean vs median after log1p (should move much closer together):
  raw     mean=5200  median=731
  log1p   mean=6.19  median=6.60


Every traffic column is heavy-tailed: impressions_90d mean (5,200) is 7x its median (731);
engagement_rate mean (2.53%) sits far above its median (0.00%) because most pages get zero
engaged sessions in a 90-day window. log1p brings impressions' mean and median much closer
(6.19 vs 6.60), confirming a log transform (or rank/grouped-median comparison) is the right
tool for every test below, not raw means on raw values.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three safe signals, each stated as a claim a content strategist would actually believe,
then tested with a grouped table and a verdict.

**Test 1 — "A higher search-volume keyword should mean more actual impressions."**
**Test 2 — "Content type affects how much readers engage once they land."**
**Test 3 — "Longer articles keep readers scrolling further."**


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
d1 = df.dropna(subset=["search_volume"]).copy()
bins = [-1, 0, 500, 5000, 10**9]
labels = ["0", "1-500", "501-5000", "5000+"]
d1["sv_bucket"] = pd.cut(d1["search_volume"], bins=bins, labels=labels)

t1 = d1.groupby("sv_bucket", observed=True).agg(
    n=("content_id", "size"),
    median_impressions=("impressions_90d", "median"),
    mean_impressions=("impressions_90d", "mean"),
)
print(t1.to_string())

               n  median_impressions  mean_impressions
sv_bucket                                             
0          11081               998.0       5919.118942
1-500      15423               865.0       5391.191013
501-5000     876               846.0       5955.060502
5000+        152               776.0       5391.644737


VERDICT: OPPOSITE.
n per bucket (11,081 / 15,423 / 876 / 152) all clear the sample floor. Median impressions_90d
does not rise with search_volume — it drifts slightly DOWN as the target keyword's search
volume goes up (998 -> 865 -> 846 -> 776). A keyword's estimated search volume is a demand
estimate for the *keyword*, not a promise about any one page's actual impressions — ranking,
query-match, and cannibalization across a client's other pages all break that link. Practical
read: don't use search_volume alone to predict a page's traffic potential.

In [ ]:
d2 = df[df["sessions_90d"] > 0].copy()
t2 = d2.groupby("content_type", observed=True).agg(
    n=("content_id", "size"),
    total_sessions=("sessions_90d", "sum"),
    total_engaged=("engaged_sessions_90d", "sum"),
)
t2["weighted_engagement_rate_pct"] = (t2["total_engaged"] / t2["total_sessions"] * 100).round(2)
print(t2.to_string())

                        n  total_sessions  total_engaged  weighted_engagement_rate_pct
content_type                                                                          
comparison article    697            2526              6                          0.24
feedly article       2096           18727            194                          1.04
keyword article     27207         1090746          29558                          2.71


VERDICT: MIXED.
Weighted by sessions (sum engaged / sum sessions, not a mean of per-row rates, per the
denominator trap) the ranking is clean: keyword article 2.71% > feedly article 1.04% >
comparison article 0.24%. n clears the floor for all three (697-27,207 rows), but comparison
article's 0.24% rests on just 6 total engaged sessions out of 2,526 — technically over the row
floor, but the underlying event count is too thin to trust as a stable rate. Practical read:
keyword articles clearly out-engage the other two types; the comparison-article number is
directional at best, not a confident number to act on yet.

In [ ]:
d3 = df.dropna(subset=["word_count", "scroll_rate"])
d3 = d3[d3["pageviews_90d"] > 0]

print("Median pageviews_90d by tier (checking for a small-denominator trap):")
print(d3.groupby("word_count_tier", observed=True)["pageviews_90d"].median().to_string())
print()

t3 = d3.groupby("word_count_tier", observed=True).agg(
    n=("content_id", "size"),
    total_scroll=("scroll_events_90d", "sum"),
    total_pv=("pageviews_90d", "sum"),
)
t3["weighted_scroll_rate"] = (t3["total_scroll"] / t3["total_pv"]).round(2)
print(t3.to_string())

Median pageviews_90d by tier (checking for a small-denominator trap):
word_count_tier
1000-2000     4.0
2000-3500     7.0
3500+        29.0
<1000         1.0

                     n  total_scroll  total_pv  weighted_scroll_rate
word_count_tier                                                     
1000-2000         3778          9433     58159                  0.16
2000-3500        11150         40042    399966                  0.10
3500+             6278         52722    789431                  0.07
<1000              973           842      2183                  0.39


VERDICT: OPPOSITE.
The <1000-word tier looks like the best scroller (weighted rate 0.39) until you check its
median pageviews_90d: 1. A rate built on 1-2 pageviews per page is noise, not signal (the
sample-size trap the skill warns about) - I'm excluding that tier from the claim. Among the
three tiers with real traffic (n=3,778 / 11,150 / 6,278, all well past the floor), the weighted
scroll rate actually falls as word count rises: 1000-2000 words = 0.16, 2000-3500 = 0.10,
3500+ = 0.07. "Longer keeps people scrolling further" is not supported - if anything, shorter
(but not razor-thin) pages hold attention slightly better per pageview. Practical read: word
count isn't a usable proxy for engagement depth in this data.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's real refresh/needs-attention flags lean on staleness as their core assumption:
a page that hasn't been touched in a while is more likely to be declining. This is the exact
signal my own baseline rule (w04_baseline_score) uses as gate #1 ("stale" = 90+ days), so it's
worth testing formally here rather than assuming the product team got it right.

Claim: pages with a longer gap since their last update are more likely to be tagged
`trend_direction == "down"`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
bins2 = [0, 30, 90, 180, 10_000]
labels2 = ["0-30", "31-90", "91-180", "181+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins2, labels=labels2)

t4 = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "size"),
    decline_rate=("trend_direction", lambda s: (s == "down").mean()),
)
print(t4.to_string())
print()
print("overall base decline rate:", round((df["trend_direction"] == "down").mean(), 3))


                      n  decline_rate
staleness_bucket                     
0-30              20480      0.511377
31-90               175      0.588571
91-180             9171      0.611057
181+                174      0.471264

overall base decline rate: 0.542


VERDICT: MIXED.
Decline rate climbs with staleness through 180 days: 0-30 days = 51.1% (n=20,480), 31-90 days =
58.9% (n=175), 91-180 days = 61.1% (n=9,171) - all above the 54.2% base rate and all past the
sample floor. But the 181+ bucket drops to 47.1% (n=174), below the base rate, on a bucket a
hundredth the size of the biggest one. Same shape I found testing this in the baseline
notebook, which is reassuring - it's a real, reproducible pattern, not noise from one run.
Staleness is a defensible signal for a "needs attention" flag up to ~180 days; I would not
trust it to mean "even more overdue = even more likely declining" past that point.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

For a content team: don't prioritize refresh work by a keyword's search-volume estimate alone -
it doesn't track with a page's real impressions in this data. Content type is a better,
if imperfect, engagement signal than page length - keyword articles reliably out-engage feedly
and comparison content, while word count itself doesn't predict scroll depth. Staleness stays
the most defensible single trigger for a review flag, but it should be capped around 180 days -
past that, "how overdue" stops being informative and the flag should lean on other signals instead.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Share of pages in the 91-180 day 'trust the staleness signal' window:",
      round((df["staleness_bucket"] == "91-180").mean(), 3))
print("Share of pages past 180 days (where the signal reverses and shouldn't be trusted alone):",
      round((df["staleness_bucket"] == "181+").mean(), 3))


Share of pages in the 91-180 day 'trust the staleness signal' window: 0.306
Share of pages past 180 days (where the signal reverses and shouldn't be trusted alone): 0.006


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.